# GPT: the decoder-only transformer (GPT-2 small, in NumPy)
1. Load the released GPT-2 small weights (`tools/gpt2_small.py`: Hugging Face safetensors by byte range, cached in ~/.cache/campusx/gpt2; about 500 MB on the first run, which needs internet).
2. Every tensor and its shape; tied unembedding; parameter count by matrix kind (GPT-2 small exact, GPT-3 from Brown et al. 2020 Table 2.1).
3. One forward pass: a prediction at every position; the loss on a real text the model never saw; the causal check.
4. Pre-LN: the residual stream grows from layer to layer; what each sub-block adds.
5. Context size, GELU, and greedy generation checked against a published output.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[2] / "tools"))
import gpt2_small as g

np.set_printoptions(precision=3, suppress=True)
DATA = Path("data"); DATA.mkdir(exist_ok=True)
W = g.load()
print("tensors:", len(W), " parameters:", f"{g.n_params(W):,}")

## 1. Every tensor of one block, and the tensors outside the blocks
Hugging Face stores GPT-2's linear layers as (input, output), so a vector times the matrix gives the output.

In [ ]:
for k, v in W.items():
    if k.startswith("h.0.") or not k.startswith("h."):
        print(f"{k:28s} {str(v.shape):14s} {v.size:>12,}")
print("blocks:", len({k.split('.')[1] for k in W if k.startswith('h.')}))
print("separate output (unembedding) matrix stored?", any("lm_head" in k for k in W))

## 2. Parameter count by kind of matrix
Q, K and V are stored side by side in `c_attn` (768 x 2304); we split them into thirds.
"Biases and layer norms" collects every bias vector and every LayerNorm gain and shift.

In [ ]:
def tally_gpt2(W):
    L = 12
    t = {"token embedding W_E (also the unembedding)": W["wte.weight"].size,
         "position embedding W_P": W["wpe.weight"].size}
    for name in ("query W_Q", "key W_K", "value W_V"):
        t[name] = L * 768 * 768
    t["attention output W_O"] = sum(W[f"h.{l}.attn.c_proj.weight"].size for l in range(L))
    t["MLP up"] = sum(W[f"h.{l}.mlp.c_fc.weight"].size for l in range(L))
    t["MLP down"] = sum(W[f"h.{l}.mlp.c_proj.weight"].size for l in range(L))
    t["biases and layer norms"] = sum(v.size for k, v in W.items() if v.ndim == 1)
    return t


def tally_formula(d, L, V, ctx):
    """Same kinds, from the sizes alone: per block 12 d^2 weights + 13 d biases/norms; tied unembedding."""
    return {"token embedding W_E (also the unembedding)": V * d, "position embedding W_P": ctx * d,
            "query W_Q": L * d * d, "key W_K": L * d * d, "value W_V": L * d * d,
            "attention output W_O": L * d * d, "MLP up": L * 4 * d * d, "MLP down": L * 4 * d * d,
            "biases and layer norms": L * 13 * d + 2 * d}


t2 = tally_gpt2(W)
assert sum(t2.values()) == g.n_params(W) == 124_439_808
assert t2 == tally_formula(768, 12, 50257, 1024)          # the formula reproduces the file exactly
t3 = tally_formula(12288, 96, 50257, 2048)                  # GPT-3 175B: Brown et al. 2020, Table 2.1
rows = [(m, k, v) for m, t in (("GPT-2 small", t2), ("GPT-3 175B", t3)) for k, v in t.items()]
tally = pd.DataFrame(rows, columns=["model", "kind", "parameters"])
tally["share_percent"] = 100 * tally.parameters / tally.groupby("model").parameters.transform("sum")
tally.to_csv(DATA / "param_tally.csv", index=False)
print(tally.to_string(index=False))
for m, t in (("GPT-2 small", t2), ("GPT-3 175B", t3)):
    tot = sum(t.values())
    attn = sum(v for k, v in t.items() if k.split()[1] in ("W_Q", "W_K", "W_V") or "output W_O" in k)
    mlp = t["MLP up"] + t["MLP down"]
    print(f"{m}: total {tot:,}; attention matrices {attn/tot:.1%}; MLP matrices {mlp/tot:.1%}; "
          f"embeddings {(t['token embedding W_E (also the unembedding)'] + t['position embedding W_P'])/tot:.1%}")
print("GPT-3 untied (a separate 50,257 x 12,288 unembedding):", f"{sum(t3.values()) + 50257 * 12288:,}")

## 3. All eight GPT-3 sizes: our count from Table 2.1 vs the paper's own counts
Table 2.1 gives n_layers, d_model, and a rounded n_params; Table D.1 gives the parameter count in millions used for the compute estimate.
All models: context 2,048, d_ff = 4 d_model, the GPT-2 vocabulary of 50,257 tokens (§2.1).

In [ ]:
gpt3 = pd.DataFrame({"model": ["Small", "Medium", "Large", "XL", "2.7B", "6.7B", "13B", "175B"],
                     "layers": [12, 24, 24, 24, 32, 32, 40, 96],
                     "d_model": [768, 1024, 1536, 2048, 2560, 4096, 5140, 12288],
                     "table_2_1": ["125M", "350M", "760M", "1.3B", "2.7B", "6.7B", "13.0B", "175.0B"],
                     "table_D_1_millions": [125, 356, 760, 1320, 2650, 6660, 12850, 174600]})
gpt3["tied_millions"] = [sum(tally_formula(d, L, 50257, 2048).values()) / 1e6 for d, L in zip(gpt3.d_model, gpt3.layers)]
gpt3["untied_millions"] = gpt3.tied_millions + 50257 * gpt3.d_model / 1e6
gpt3.to_csv(DATA / "gpt3_sizes.csv", index=False)
print(gpt3.round(1).to_string(index=False))
# 13B: Table 2.1 says d_model 5140 with 40 heads of 128, but 40 x 128 = 5120
print("13B with d_model = 5120:", round(sum(tally_formula(5120, 40, 50257, 2048).values()) / 1e6, 1), "million")

## 4. GPT-2 small vs the "117M" of Radford et al. 2019, Table 2
The paper says its smallest model "is equivalent to the original GPT". GPT-1 used a 40,000-merge BPE vocabulary and 512-token sequences (Radford et al. 2018, §4.1).
Same 12 blocks of 768, with about 40,000 tokens and 512 positions:

In [ ]:
body = sum(v for k, v in t2.items() if "embedding" not in k)
for V, ctx in ((40_000, 512), (40_478, 512), (50_257, 1024)):
    print(f"vocabulary {V:,}, positions {ctx}: {(body + V * 768 + ctx * 768)/1e6:.1f}M")

## 5. Tokens

In [ ]:
tok = g.tokenizer()
print("vocabulary size:", tok.get_vocab_size())
for s in ["Steve Jobs was the founder of", "Unbelievably, tokenization splits rare words."]:
    ids = g.encode(s)
    print(len(ids), [g.decode([i]) for i in ids], ids)

## 6. One forward pass: a prediction at every position

In [ ]:
prompt = "Steve Jobs was the founder of"
ids = g.encode(prompt)
out = g.forward(ids, W)
P = g.softmax(out["logits"])
rows = []
for i in range(len(ids)):
    top = np.argsort(-P[i])[:3]
    nxt = g.decode([ids[i + 1]]) if i + 1 < len(ids) else ""
    rows.append(dict(position=i, token=g.decode([ids[i]]), next_actual=nxt,
                     p_actual=float(P[i, ids[i + 1]]) if nxt else np.nan,
                     **{f"top{j+1}": g.decode([t]) for j, t in enumerate(top)},
                     **{f"p{j+1}": float(P[i, t]) for j, t in enumerate(top)}))
every = pd.DataFrame(rows)
every.to_csv(DATA / "every_position.csv", index=False)
print(every.round(3).to_string(index=False))

### The loss on a real text the model never saw
The first sentences of the GPT-2 paper's abstract (published February 2019; WebText holds no links created after December 2017, Radford et al. 2019 §2.1).

In [ ]:
text = ("Natural language processing tasks, such as question answering, machine translation, reading comprehension, "
        "and summarization, are typically approached with supervised learning on task-specific datasets. We demonstrate "
        "that language models begin to learn these tasks without any explicit supervision when trained on a new dataset "
        "of millions of webpages called WebText.")
tid = g.encode(text)
o = g.forward(tid, W)
lp = np.log(g.softmax(o["logits"][:-1]))
nll = -lp[np.arange(len(tid) - 1), tid[1:]]
top1 = (o["logits"][:-1].argmax(-1) == np.array(tid[1:])).mean()
print(f"{len(tid)} tokens -> {len(tid)-1} predictions in ONE pass")
print(f"mean cross-entropy {nll.mean():.3f} nats (uniform guess: ln 50257 = {np.log(50257):.3f}); "
      f"perplexity {np.exp(nll.mean()):.1f}; top-1 correct {top1:.1%}")
easy = np.argsort(nll)[:5]; hard = np.argsort(-nll)[:5]
print("easiest:", [(g.decode(tid[:i + 1][-3:]), g.decode([tid[i + 1]]), round(float(nll[i]), 3)) for i in easy])
print("hardest:", [(g.decode(tid[:i + 1][-3:]), g.decode([tid[i + 1]]), round(float(nll[i]), 2)) for i in hard])

### Causal check: changing the last token changes no earlier prediction

In [ ]:
ids2 = ids[:-1] + g.encode(" in")
o2 = g.forward(ids2, W)
print("largest change at positions 0..4:", np.abs(o2["logits"][:-1] - out["logits"][:-1]).max(),
      "  at the last position:", np.abs(o2["logits"][-1] - out["logits"][-1]).max())
print("attention above the diagonal (future keys), largest weight:", np.triu(out["attn"], 1).max())

## 7. Pre-LN: the residual stream and what each sub-block adds
x <- x + Attn(LN1(x)); x <- x + MLP(LN2(x)); after block 12, ln_f, then the unembedding.

In [ ]:
r = out["resid"]
norms = pd.DataFrame({"layer": range(13), "stream_norm_last": np.linalg.norm(r[:, -1], axis=-1),
                      "stream_norm_mean_rest": np.linalg.norm(r[:, 1:], axis=-1).mean(1)})
norms["attn_add_last"] = [np.linalg.norm(out["attn_out"][l, -1]) for l in range(12)] + [np.nan]
norms["mlp_add_last"] = [np.linalg.norm(out["mlp_out"][l, -1]) for l in range(12)] + [np.nan]
norms["cos_before_after_block"] = [float(r[l, -1] @ r[l + 1, -1] / np.linalg.norm(r[l, -1]) / np.linalg.norm(r[l + 1, -1]))
                                   for l in range(12)] + [np.nan]
norms.to_csv(DATA / "resid_norm.csv", index=False)
print(norms.round(2).to_string(index=False))
print("norm of position 0 (first token) by layer:", np.linalg.norm(r[:, 0], axis=-1).round(0))
ln_in = g.layer_norm(r[5, -1], W["h.5.ln_1.weight"] * 0 + 1, W["h.5.ln_1.bias"] * 0)
print("LayerNorm input to block 6 (before gain): mean", ln_in.mean().round(4), "std", ln_in.std().round(3))
print("final vector after ln_f: norm", np.linalg.norm(out["final"][-1]).round(1))

Without the final layer norm, does the prediction survive?

In [ ]:
z_noln = g.unembed(r[-1, -1], W)
print("top-5 with ln_f   :", [g.decode([i]) for i in np.argsort(-out["logits"][-1])[:5]])
print("top-5 without ln_f:", [g.decode([i]) for i in np.argsort(-z_noln)[:5]])

## 8. Context size
The position table has 1,024 rows, so 1,024 tokens is the longest input GPT-2 can read.

In [ ]:
print("position table:", W["wpe.weight"].shape, " config n_ctx:", g.CONFIG["n_ctx"])
for n in (128, 1024, 2048):
    print(f"n = {n:5d}: attention weights per head {n*n:>10,}; GPT-2 (144 heads) {144*n*n:>13,}; GPT-3 (96 x 96 heads) {9216*n*n:>15,}")
try:
    g.forward(list(range(1025)), W)
except AssertionError:
    print("1,025 tokens: refused (no position vector for position 1,025)")

## 9. GELU vs ReLU (GPT-2 config: activation_function = gelu_new)

In [ ]:
from math import erf
xs = np.linspace(-4, 3, 8)
exact = np.array([0.5 * x * (1 + erf(x / np.sqrt(2))) for x in xs])
print(pd.DataFrame({"x": xs, "ReLU": np.maximum(xs, 0), "GELU (exact, x Phi(x))": exact, "gelu_new (tanh)": g.gelu(xs)}).round(4).to_string(index=False))
xx = np.linspace(-6, 6, 2001)
print("largest |gelu_new - exact| on [-6, 6]:", np.abs(g.gelu(xx) - np.array([0.5 * x * (1 + erf(x / np.sqrt(2))) for x in xx])).max())
print("GELU minimum:", g.gelu(xx).min().round(4), "at x =", xx[g.gelu(xx).argmin()].round(3))
acts = out["mlp_pre"].ravel()
print(f"share of MLP inputs below 0 on the prompt: {(acts < 0).mean():.1%}; their GELU outputs lie in [{g.gelu(acts[acts<0]).min():.3f}, 0]")

## 10. Generation: predict, pick, append, repeat
Greedy (always the top token) on the prompt of the Hugging Face blog "How to generate text" (von Platen 2020), which shows GPT-2's greedy output.

In [ ]:
gen = g.generate(g.encode("I enjoy walking with my cute dog"), W, n=30)
print(repr(g.decode(gen)))
steps = []
ids_g = g.encode(prompt)
for s in range(4):
    z = g.forward(ids_g, W)["logits"][-1]; p = g.softmax(z); top = np.argsort(-p)[:3]
    steps.append(dict(step=s + 1, text=g.decode(ids_g), **{f"top{j+1}": g.decode([t]) for j, t in enumerate(top)},
                      **{f"p{j+1}": float(p[t]) for j, t in enumerate(top)}))
    ids_g = ids_g + [int(top[0])]
steps = pd.DataFrame(steps)
steps.to_csv(DATA / "greedy_steps.csv", index=False)
print(steps.round(3).to_string(index=False))